<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_12/fl/local_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline: Nur lokales Training (ohne Federated Learning) 🔬

Dieses **optionale, dritte Notebook** ist kein Teil des eigentlichen Federated-Learning-Experiments
— du brauchst dafür keine Verbindung zum Server und kein `SERVER_ADDRESS`. Stattdessen trainierst du
ein Modell **ausschließlich auf deinem eigenen kleinen Datenanteil** (derselben Partition, die du
auch als Flower-Client in `client.ipynb` benutzt hast) und beobachtest, wie gut dieses Modell im
Vergleich zum gemeinsam per Federated Learning trainierten globalen Modell abschneidet.

## Warum ist das interessant?

Als einzelner Client siehst du nur einen kleinen Ausschnitt von MNIST (bei 20 Clients z. B. nur
1/20 der Trainingsbilder, bei Non-IID-Partitionierung sogar überwiegend nur ein oder zwei Ziffern).
Ein Modell, das **nur** auf diesen Daten trainiert, kann daher kaum alle zehn Ziffern zuverlässig
erkennen. Das federated trainierte Modell hingegen hat — ohne dass jemals Rohdaten geteilt wurden —
indirekt vom Wissen **aller** Clients profitiert, weil in jeder Runde die lokal gelernten Gewichte
aller Clients gemittelt wurden.

## Fairer Vergleich

Damit der Vergleich fair ist, trainierst du hier **genauso viele lokale Epochen insgesamt**, wie du
auch als federated Client durchlaufen hast:

```
GESAMT_EPOCHEN = num_rounds × local_epochs
```

Bei den Standardwerten aus `server.ipynb` (`num_rounds=10`, `local_epochs=1`) sind das also
**10 Epochen** — der einzige Unterschied ist, dass hier **keine Aggregation** mit anderen Clients
stattfindet.

## 1. Installation

In [ ]:
!pip install -q tensorflow matplotlib numpy

## 2. Gemeinsame Hilfsfunktionen (`common.py`)

Identisch zu den anderen beiden Notebooks.

In [ ]:
!wget -O common.py https://raw.githubusercontent.com/dgaida/wpf_dlml_th_public/main/assets/exercises/week_12/fl/common.py

## 3. Konfiguration — hier anpassen

Verwende **dieselbe** `CLIENT_ID` und `PARTITION_STRATEGY` wie in deinem `client.ipynb`, damit du
wirklich deine eigene Datenpartition mit dem federated Ergebnis vergleichst. Trage außerdem die
**finale globale Accuracy** ein, die am Ende von `server.ipynb` ausgegeben wurde (letzte
`Global Accuracy`-Zeile) — dann zeichnen die Plots unten automatisch eine Vergleichslinie.

In [ ]:
# -----------------------------------------------------------------------
# Dieselben Werte wie in client.ipynb verwenden:
# -----------------------------------------------------------------------
CLIENT_ID = 0                    # <-- deine individuelle Client-ID (0-19)
PARTITION_STRATEGY = "iid"       # <-- muss mit client.ipynb / server.ipynb übereinstimmen

# Anzahl lokaler Trainings-Epochen für den fairen Vergleich:
# GESAMT_EPOCHEN = num_rounds * local_epochs aus server.ipynb (Standard: 10 * 1 = 10)
TOTAL_EPOCHS = 10

# Optional: die finale "Global Accuracy" aus der letzten Zeile der Server-Ausgabe eintragen,
# um eine Vergleichslinie/-balken zu erhalten. None lassen, falls (noch) nicht bekannt.
FEDERATED_ACCURACY = None  # z. B. 0.9521


## 4. Daten laden

Dieselbe Partition wie in `client.ipynb` — nur wird sie hier nie an einen Server geschickt.

In [ ]:
import common

x_train, y_train = common.load_client_data(CLIENT_ID, strategy=PARTITION_STRATEGY)
x_test, y_test = common.load_test_data()

print(
    f"Client {CLIENT_ID}: {x_train.shape[0]} lokale Trainingsbilder "
    f"(Strategie='{PARTITION_STRATEGY}') -- nur diese werden für das Training verwendet."
)

## 5. Optional: eigene Klassenverteilung

In [ ]:
import matplotlib.pyplot as plt

common.plot_distribution(y_train, client_id=CLIENT_ID)
plt.show()


## 6. Modell

Dieselbe Architektur wie in `server.ipynb` / `client.ipynb`, damit die Modellkapazität
identisch ist und nur der Trainingsmodus (federated vs. rein lokal) den Unterschied ausmacht.

In [ ]:
local_model = common.create_model()
local_model.summary()

## 7. Rein lokales Training

Anders als beim Flower-Client gibt es hier **keine Kommunikation** mit einem Server und **keine
Aggregation** — das Modell sieht während des gesamten Trainings ausschließlich deine eigene
Partition. Nach jeder Epoche wird die Accuracy auf dem globalen Testdatensatz gemessen, damit die
Kurve direkt mit der federated Accuracy vergleichbar ist.

In [ ]:
history = local_model.fit(
    x_train,
    y_train,
    epochs=TOTAL_EPOCHS,
    batch_size=common.FLConfig().batch_size,
    validation_data=(x_test, y_test),
    verbose=1,
)

final_local_accuracy = history.history["val_accuracy"][-1]
print(f"\nFinale Test-Accuracy (nur lokales Training): {final_local_accuracy:.4f}")


## 8. Visualisierung: lokales Training vs. Federated Learning

Die rote Kurve zeigt, wie sich dein rein lokal trainiertes Modell über die Epochen entwickelt. Die
blau gestrichelte Linie (falls `FEDERATED_ACCURACY` gesetzt wurde) zeigt zum Vergleich die finale
Accuracy des gemeinsam trainierten federated Modells.

In [ ]:
import os

epochs = list(range(1, TOTAL_EPOCHS + 1))
val_accuracies = history.history["val_accuracy"]

fig = common.plot_local_training_curve(epochs, val_accuracies, federated_accuracy=FEDERATED_ACCURACY)

# Create the 'figures' directory if it doesn't exist
os.makedirs("figures", exist_ok=True)

fig.savefig("figures/local_vs_federated_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Direkter Vergleich als Balkendiagramm

Nur sinnvoll, wenn `FEDERATED_ACCURACY` in Abschnitt 3 gesetzt wurde.

In [ ]:
if FEDERATED_ACCURACY is not None:
    fig = common.plot_local_vs_federated_bar(final_local_accuracy, FEDERATED_ACCURACY)
    fig.savefig("figures/local_vs_federated_bar.png", dpi=150, bbox_inches="tight")
    plt.show()

    diff = FEDERATED_ACCURACY - final_local_accuracy
    print(
        f"Federated Learning erreicht eine um {diff:.4f} "
        f"({diff * 100:.1f} Prozentpunkte) höhere Accuracy als rein lokales Training."
    )
else:
    print("FEDERATED_ACCURACY ist nicht gesetzt -- trage sie in Abschnitt 3 ein für den Vergleich.")


## Fazit & zum Weiterdenken

- Bei **IID**-Partitionierung ist der Unterschied meist moderat, da jeder Client bereits eine
  einigermaßen repräsentative Mischung aller Ziffern sieht.
- Bei **Non-IID**-Partitionierung (`"shard"` oder `"dominant"`) fällt der Unterschied in der Regel
  deutlich größer aus: Dein lokales Modell hat kaum eine Chance, Ziffern zuverlässig zu erkennen,
  die es nie oder kaum gesehen hat — das federated Modell dagegen schon, weil es indirekt vom
  Wissen aller anderen Clients profitiert.
- **Zum Ausprobieren:** `PARTITION_STRATEGY` in Abschnitt 3 auf `"shard"` oder `"dominant"` ändern,
  Notebook erneut ausführen (mit passendem `FEDERATED_ACCURACY`-Wert aus einem entsprechenden
  Non-IID-Durchlauf von `server.ipynb`) und den Unterschied vergrößern beobachten.